## Upload CIFAR-10 to S3 in 20 Parts

### Objective
Build a local CIFAR-10 zip archive if needed, split it into multiple pieces, upload all pieces to S3 through LAILA in one batched `GroupFuture`, and store the manifest as another S3-backed LAILA entry.

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Objective
Set up imports, notebook paths, and a few constants we will reuse across cells.

In [9]:
import hashlib
import json
import math
import sys
import tarfile
import urllib.request
import zipfile
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path("/home/ubuntu/laila")
PYTHON_ROOT = PROJECT_ROOT.parent
if str(PYTHON_ROOT) not in sys.path:
    sys.path.append(str(PYTHON_ROOT))

import laila
laila.read_args(PROJECT_ROOT / "vault" / "dev_secrets.toml")
from laila.pool import S3Pool

OFFICIAL_CIFAR10_TAR_URL = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
POOL_NICKNAME = "cifar_10_s3_pool"
MANIFEST_NICKNAME = "cifar_10_s3_manifest"
NUM_PARTS = 20
ARCHIVE_PATH = PROJECT_ROOT / "examples" / "training" / "cifar" / "cifar-10-python.zip"

### Objective
Define helper functions for preparing the archive, constructing the S3 pool, and splitting the binary payload into `numpy.uint8` chunks that LAILA can wrap safely.

In [10]:
def ensure_cifar10_zip(zip_path: Path) -> Path:
    # Reuse an existing zip if it is already available locally.
    if zip_path.exists():
        return zip_path

    # Otherwise download the official tar.gz archive first.
    tar_path = zip_path.with_suffix(".tar.gz")
    if not tar_path.exists():
        print(f"Downloading CIFAR-10 archive to {tar_path} ...")
        urllib.request.urlretrieve(OFFICIAL_CIFAR10_TAR_URL, tar_path)

    # Convert the tarball into a zip so the downstream training example
    # reconstructs a single archive file with a clear manifest target.
    print(f"Converting {tar_path.name} -> {zip_path.name} ...")
    with tarfile.open(tar_path, "r:gz") as tar, zipfile.ZipFile(
        zip_path, "w", compression=zipfile.ZIP_DEFLATED
    ) as zf:
        for member in tar.getmembers():
            if not member.isfile():
                continue
            extracted = tar.extractfile(member)
            if extracted is None:
                continue
            zf.writestr(member.name, extracted.read())

    return zip_path


def build_s3_pool() -> S3Pool:
    required = ["AWS_BUCKET_NAME", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_REGION"]
    missing = [name for name in required if getattr(laila.args, name, None) is None]
    if missing:
        raise RuntimeError(f"Missing laila.args values: {', '.join(missing)}")

    return S3Pool(
        bucket_name=laila.args.AWS_BUCKET_NAME,
        access_key_id=laila.args.AWS_ACCESS_KEY_ID,
        secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
        region_name=laila.args.AWS_REGION,
    )


def manifest_entry_id_from_nickname(manifest_nickname: str) -> str:
    # Nicknames deterministically map to entry IDs, so we can re-derive this later.
    return laila.constant(data=np.zeros(1, dtype=np.uint8), nickname=manifest_nickname).global_id


def split_bytes_into_arrays(payload: bytes, num_parts: int) -> tuple[list[np.ndarray], list[int]]:
    # Use equal-sized chunks and store each chunk as uint8 data.
    part_size = math.ceil(len(payload) / num_parts)
    arrays = []
    sizes = []
    for index in range(num_parts):
        start = index * part_size
        stop = min(len(payload), start + part_size)
        chunk = payload[start:stop]
        if not chunk:
            continue
        arrays.append(np.frombuffer(chunk, dtype=np.uint8).copy())
        sizes.append(len(chunk))
    return arrays, sizes

### Objective
Create or load the CIFAR-10 zip archive, compute a deterministic content hash, and split it into 20 parts.

In [11]:
archive_path = ensure_cifar10_zip(ARCHIVE_PATH)
archive_bytes = archive_path.read_bytes()
archive_sha256 = hashlib.sha256(archive_bytes).hexdigest()
chunk_arrays, chunk_sizes = split_bytes_into_arrays(archive_bytes, NUM_PARTS)

print("Archive:", archive_path)
print("Archive size (bytes):", len(archive_bytes))
print("SHA256:", archive_sha256)
print("Prepared parts:", len(chunk_arrays))

Archive: /home/ubuntu/laila/examples/training/cifar/cifar-10-python.zip
Archive size (bytes): 170022227
SHA256: 5f31aa05e6757567083495c0eb00025e996fa60cfc56aa48abe1709f6a645194
Prepared parts: 20


### Objective
Register an S3 pool and wrap every archive chunk as a separate deterministic LAILA entry so the upload can happen through a `GroupFuture`.

In [12]:
s3_pool = build_s3_pool()
laila.add_pool(s3_pool, pool_nickname=POOL_NICKNAME)

# Each part gets its own entry. The nickname makes reruns easier to track.
part_entries = [
    laila.constant(data=chunk_array, nickname=f"cifar10-zip-part-{archive_sha256[:12]}-{index:02d}")
    for index, chunk_array in enumerate(chunk_arrays)
]

print("Entry count:", len(part_entries))
print("First entry id:", part_entries[0].global_id)

Entry count: 20
First entry id: LAILA:ENTRY:GLOBAL_ID:6c97316e-5b8f-5b7a-a31a-d78828adf61b


### Objective
Upload all parts to S3 in one call, wait on the resulting `GroupFuture`, and inspect the final status.

In [13]:
# Batched upload: LAILA returns a GroupFuture because we pass a list of entries.
upload_group_future = laila.memorize(part_entries, pool_nickname=POOL_NICKNAME)
print("Upload future type:", type(upload_group_future).__name__)
print("Status before wait:", upload_group_future.status)

upload_group_future.wait()
print("Status after wait:", upload_group_future.status)
print("Underlying futures:", len(upload_group_future))

Upload future type: GroupFuture
Status before wait: {'total': 20.0, 'percentages': {'finished': 0.0, 'running': 0.0, 'not_started': 100.0, 'error': 0.0, 'cancelled': 0.0}}
Status after wait: {'total': 20.0, 'percentages': {'finished': 100.0, 'running': 0.0, 'not_started': 0.0, 'error': 0.0, 'cancelled': 0.0}}
Underlying futures: 20


### Objective
Build a manifest that records the archive hash, pool nickname, and ordered entry IDs, then memorize that manifest onto S3 as its own LAILA entry for the training notebook.

In [14]:
manifest_entry_id = manifest_entry_id_from_nickname(MANIFEST_NICKNAME)
manifest = {
    "archive_path": str(archive_path),
    "archive_name": archive_path.name,
    "archive_size_bytes": len(archive_bytes),
    "archive_sha256": archive_sha256,
    "num_parts": len(part_entries),
    "chunk_sizes": chunk_sizes,
    "pool_nickname": POOL_NICKNAME,
    "manifest_nickname": MANIFEST_NICKNAME,
    "manifest_entry_id": manifest_entry_id,
    "entry_ids": [entry.global_id for entry in part_entries],
}

manifest_entry = laila.constant(data=manifest, nickname=MANIFEST_NICKNAME)
manifest_future = laila.memorize(manifest_entry, pool_nickname=POOL_NICKNAME)
print("Manifest future type:", type(manifest_future).__name__)
manifest_future.wait()

print("Manifest upload complete.")
print("Manifest entry id:", manifest_entry.global_id)
print(json.dumps(manifest, indent=2)[:600] + "\n...")

Manifest future type: ConcurrentPackageFuture
Manifest upload complete.
Manifest entry id: LAILA:ENTRY:GLOBAL_ID:5c417b08-7228-5ac8-90f5-0298469122bf
{
  "archive_path": "/home/ubuntu/laila/examples/training/cifar/cifar-10-python.zip",
  "archive_name": "cifar-10-python.zip",
  "archive_size_bytes": 170022227,
  "archive_sha256": "5f31aa05e6757567083495c0eb00025e996fa60cfc56aa48abe1709f6a645194",
  "num_parts": 20,
  "chunk_sizes": [
    8501112,
    8501112,
    8501112,
    8501112,
    8501112,
    8501112,
    8501112,
    8501112,
    8501112,
    8501112,
    8501112,
    8501112,
    8501112,
    8501112,
    8501112,
    8501112,
    8501112,
    8501112,
    8501112,
    8501099
  ],
  "pool_nickname": "cifar_10_s3_pool",
  "manife
...
